In [1]:
import pandas as pd

df = pd.read_csv("bank_churn.csv")
print(df.shape)
print(df[["Geography", "Exited"]].head())

(10000, 18)
  Geography  Exited
0    France       1
1     Spain       0
2    France       1
3    France       0
4     Spain       0


In [2]:
churn_by_country = df.groupby("Geography")["Exited"].mean() * 100
churn_by_country = churn_by_country.reset_index()
churn_by_country.columns = ["Country", "Churn Rate (%)"]
churn_by_country = churn_by_country.sort_values("Churn Rate (%)", ascending=False)
print(churn_by_country)

   Country  Churn Rate (%)
1  Germany       32.443204
2    Spain       16.673395
0   France       16.174711


In [3]:
churn_summary = churn_by_country.copy()
churn_summary["Churn Rate (%)"] = churn_summary["Churn Rate (%)"].round(2)
churn_summary.to_csv("churn_summary.csv", index=False)
print("Saved! Here is your summary:")
print(churn_summary)

Saved! Here is your summary:
   Country  Churn Rate (%)
1  Germany           32.44
2    Spain           16.67
0   France           16.17


# Bank Customer Churn Analysis

## Objective
Analyze 10,000+ bank customer records to identify which country 
has the highest customer churn rate.

## Tools Used
- Python (pandas) — data loading and analysis
- SQL (SQLite) — querying and validating results
- Power BI — data visualization

## Key Finding
Germany has the highest churn rate at 32.44%, which is nearly 
2x higher than France (16.17%) and Spain (16.67%).

## Business Insight
The bank should investigate why German customers are leaving 
at twice the rate of other regions. Possible factors could 
include pricing, competition, or customer service differences 
in that market.


In [4]:
high_balance = {
    "Country": ["Germany", "France", "Spain"],
    "High_Balance_Count": [1968, 1908, 927],
    "Churn_Rate_Pct": [35.87, 17.98, 17.58]
}

df_high_balance = pd.DataFrame(high_balance)
df_high_balance.to_csv("high_balance_churn.csv", index=False)
print(df_high_balance)

   Country  High_Balance_Count  Churn_Rate_Pct
0  Germany                1968           35.87
1   France                1908           17.98
2    Spain                 927           17.58


In [5]:
import pandas as pd

churn_summary = pd.read_csv("churn_summary.csv")
high_balance = pd.read_csv("high_balance_churn.csv")

with pd.ExcelWriter("churn_analysis.xlsx", engine="openpyxl") as writer:
    churn_summary.to_excel(writer, sheet_name="Churn by Country", index=False)
    high_balance.to_excel(writer, sheet_name="High Balance Churn", index=False)

print("Excel file saved!")

Excel file saved!


In [6]:
df["Age Group"] = pd.cut(df["Age"], bins=[0,30,40,50,60,100], 
                          labels=["<30","30-40","40-50","50-60","60+"])
age_churn = df.groupby("Age Group")["Exited"].mean() * 100
age_churn = age_churn.reset_index()
age_churn.columns = ["Age Group", "Churn Rate (%)"]
age_churn["Churn Rate (%)"] = age_churn["Churn Rate (%)"].round(2)
age_churn.to_csv("age_churn.csv", index=False)
print(age_churn)

  Age Group  Churn Rate (%)
0       <30            7.52
1     30-40           12.11
2     40-50           33.97
3     50-60           56.21
4       60+           24.78


In [7]:
gender_churn = df.groupby("Gender")["Exited"].mean() * 100
gender_churn = gender_churn.reset_index()
gender_churn.columns = ["Gender", "Churn Rate (%)"]
gender_churn["Churn Rate (%)"] = gender_churn["Churn Rate (%)"].round(2)
gender_churn.to_csv("gender_churn.csv", index=False)
print(gender_churn)

   Gender  Churn Rate (%)
0  Female           25.07
1    Male           16.47


In [ ]:
with pd.ExcelWriter("churn_analysis.xlsx", engine="openpyxl") as writer:
    churn_summary.to_excel(writer, sheet_name="Churn by Country", index=False)
    high_balance.to_excel(writer, sheet_name="High Balance Churn", index=False)
    age_churn.to_excel(writer, sheet_name="Churn by Age Group", index=False)
    gender_churn.to_excel(writer, sheet_name="Churn by Gender", index=False)

print("Excel file updated with all 4 sheets!")

Excel file updated with all 4 sheets!


In [9]:
# Data Cleaning & Descriptive Statistics
print("Missing values:")
print(df.isnull().sum())

print("\nBasic Statistics:")
print(df[["Age", "Balance", "CreditScore", "Exited"]].describe().round(2))

Missing values:
RowNumber             0
CustomerId            0
Surname               0
CreditScore           0
Geography             0
Gender                0
Age                   0
Tenure                0
Balance               0
NumOfProducts         0
HasCrCard             0
IsActiveMember        0
EstimatedSalary       0
Exited                0
Complain              0
Satisfaction Score    0
Card Type             0
Point Earned          0
Age Group             0
dtype: int64

Basic Statistics:
            Age    Balance  CreditScore   Exited
count  10000.00   10000.00     10000.00  10000.0
mean      38.92   76485.89       650.53      0.2
std       10.49   62397.41        96.65      0.4
min       18.00       0.00       350.00      0.0
25%       32.00       0.00       584.00      0.0
50%       37.00   97198.54       652.00      0.0
75%       44.00  127644.24       718.00      0.0
max       92.00  250898.09       850.00      1.0


In [11]:
from scipy import stats

# Hypothesis Test: Is Germany's churn rate significantly higher?
germany = df[df["Geography"] == "Germany"]["Exited"]
others = df[df["Geography"] != "Germany"]["Exited"]

t_stat, p_value = stats.ttest_ind(germany, others)

print("Hypothesis Test: Germany vs Other Countries Churn Rate")
print(f"Germany mean churn rate: {germany.mean():.4f}")
print(f"Others mean churn rate: {others.mean():.4f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("\nResult: Germany's churn rate is STATISTICALLY SIGNIFICANTLY higher")
else:
    print("\nResult: No significant difference found")

Hypothesis Test: Germany vs Other Countries Churn Rate
Germany mean churn rate: 0.3244
Others mean churn rate: 0.1634
T-statistic: 17.5958
P-value: 0.0000

Result: Germany's churn rate is STATISTICALLY SIGNIFICANTLY higher


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Encode categorical variables
df_model = df[["Geography", "Gender", "Age", "Balance", "CreditScore", "Exited"]].copy()
df_model["Geography"] = LabelEncoder().fit_transform(df_model["Geography"])
df_model["Gender"] = LabelEncoder().fit_transform(df_model["Gender"])

# Split data
X = df_model.drop("Exited", axis=1)
y = df_model["Exited"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Results
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Logistic Regression Accuracy: 79.35%

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.97      0.88      1607
           1       0.39      0.09      0.14       393

    accuracy                           0.79      2000
   macro avg       0.60      0.53      0.51      2000
weighted avg       0.73      0.79      0.74      2000



In [13]:
# Summary of all findings
print("=" * 50)
print("BANK CUSTOMER CHURN ANALYSIS - KEY FINDINGS")
print("=" * 50)
print(f"\n1. Germany has the highest churn rate: 32.44%")
print(f"   vs Spain (16.67%) and France (16.17%)")
print(f"\n2. Hypothesis Test: Germany's higher churn rate")
print(f"   is statistically significant (p < 0.0001)")
print(f"\n3. High-balance customers in Germany churn")
print(f"   even more: 35.87%")
print(f"\n4. Female customers churn more than males")
print(f"\n5. Customers aged 50-60 have highest churn rate")
print(f"\n6. Logistic Regression Model Accuracy: 79.35%")
print("=" * 50)

BANK CUSTOMER CHURN ANALYSIS - KEY FINDINGS

1. Germany has the highest churn rate: 32.44%
   vs Spain (16.67%) and France (16.17%)

2. Hypothesis Test: Germany's higher churn rate
   is statistically significant (p < 0.0001)

3. High-balance customers in Germany churn
   even more: 35.87%

4. Female customers churn more than males

5. Customers aged 50-60 have highest churn rate

6. Logistic Regression Model Accuracy: 79.35%
